In [12]:
import requests
import urllib3
from bs4 import BeautifulSoup
import pandas as pd


#### disable warning for url ####
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#### proxy settings 
proxy = {
    'http': 'http://dcproxy.gm.com:8080',
    'https': 'http://dcproxy.gm.com:8080',
}

### Collect data from IBM web site #####

''' source: https://www.ibm.com/support/pages/recommended-fixes-hmc-code-upgrades'''
IBMurl = "https://www.ibm.com/support/pages/recommended-fixes-hmc-code-upgrades"

### getting content from IBM web site
IBMresponse = requests.get(IBMurl, proxies=proxy)

# parse content
soup = BeautifulSoup(IBMresponse.text, 'html.parser')

# find tabel from IBM website and create dataframe
table = soup.find('table')
df = pd.read_html(str(table))[0] 
df_cleaned = df.dropna().reset_index(drop=True)  # remove NaN date from dataframe
df_cleaned.columns = df_cleaned.iloc[0]          # describe columns name
df_cleaned = df_cleaned[1:]                      # remove first line after creating columns. cleaned 



#### Collect data from CMC ######
CMCurl = 'https://gm.powercmc.com/api/public/v2/inventory/cmc/ManagementConsole'
headers = {
    'X-CMC-Client-Id': 'af8204d5-6f4f-41e0-980a-f9e3daad605d',
    'X-CMC-Client-Secret': '84de9b84-30b5-40da-bf1c-03bbf62f3727',
}



CMCresponse = requests.get(CMCurl, headers=headers, proxies=proxy, verify=False)

for liste in CMCresponse.json()['HMCs']:
    print("##########")
    print(liste['Name']+":"+liste['MTMS']['SerialNumber']+":V"+liste['VersionInfo']['Version']+"R"+liste['VersionInfo']['Release']+"."+liste['VersionInfo']['ServicePackName']+"."+liste['VersionInfo']['Minor'])

##########
dcmipphhmc001.edc.nam.gm.com:215082D:V9R2.953.4
##########
dcmipphhmc002.edc.nam.gm.com:215083D:V9R2.953.4
##########
dcwipphhmc005.edc.nam.gm.com:215084D:V9R2.953.4
##########
dcwipphhmc004.edc.nam.gm.com:215085D:V9R2.953.4
##########
dcmirphhmc001.rls.mpg.nam.gm.com:130PLXA:V10R2.1041.2
##########
dcmidphhmc002.edc.nam.gm.com:130PPDA:V10R2.1031.1
##########
dcwitphhmc001.edc.nam.gm.com:130R9KA:V10R2.1031.1
##########
dcwipphhmc006.edc.nam.gm.com:130RGWA:V10R2.1031.1
##########
dcmipphhmc003.edc.nam.gm.com:130RH9A:V10R2.1031.1
##########
dcmipphhmc006.edc.nam.gm.com:13119VA:V10R2.1031.1
##########
dcmipphhmc005.edc.nam.gm.com:1311AMA:V10R2.1031.1
##########
dcwipphhmc008:0289E3D:V10R2.1031.1
##########
dcwipphhmc009:0289E4D:V10R2.1031.1
##########
dcmirvmhmc001:4e14b07:V10R2.1041.2
##########
dcwipvmhmc009:761410d:V10R2.1031.1
##########
dcmidvmhmc001.edc.nam.gm.com:2e44d05:V10R2.1031.1
##########
dcmipvmhmc005:2741d0c:V10R2.1031.1
##########
dcwitvmhmc001.edc.nam.gm.com:22

In [13]:
df_cleaned

,Target Version,Upgrade From,Upgrade Instructions,Updates,Date Available,End of Service,Models supported,POWER Support
1,V10 R2,V9 R2 and later,ppc x86,link,12/08/2022,04/30/2025,7063-CR1 7063-CR2,"P10, P9, P8"
2,V10 R1,V9 R1 and later,ppc x86,link,09/17/2021,04/30/2024,7063-CR1 7063-CR2,"P10, P9, P8"
3,V9 R2,V9 R1,ppc CR9 x86,link,11/20/2020,04/30/2023,7063-CR1 7063-CR2 7042-CR9,"P9, P8, P7"
4,V9 R1 M9xx,V8R8.6 and later,x86 ppc,link,3/20/2018,09/30/2021,"7042-CR7, CR8, CR9, OE1, and OE2 7063-CR1","P9, P8, P7"
5,V8.8.7,V8R8.5 and later,link,link,09/15/2017,08/31/2019,"7042-CR7, CR8, CR9, OE1, and OE2 7063-CR1","P8, P7, P6"
6,V8.8.6,V8.8.4 and later,link,link,11/18/2016,10/31/2018,CR5-CR9 C08,"P8, P7 P6"
7,V8.8.5,V8.8.3 and later,link,link,05/27/2016,05/31/2018,CR5-CR9 C08,"P8, P7 P6"
8,V8.8.4,V8.8.2 and later,link,link,11/20/2015,11/30/2017,CR5-CR9 C08,"P8, P7 P6"
9,V8.8.3,V8.8.1 and later,link,link,6/05/2015,7/31/2017,CR5-CR8 C08,"P8, P7 P6"
10,V8.8.2,V7.7.9 and later,link,link,11/18/2014,10/31/2016,CR5-CR8 C08,"P8, P7 P6"


In [19]:
data = []
for hmc in CMCresponse.json()['HMCs']:

    row = {
        'Name': hmc['Name'],
        'SerialNumber': hmc['MTMS']['SerialNumber'],
        'Version': f"V{hmc['VersionInfo']['Version']}R{hmc['VersionInfo']['Release']}",
        'ServicePackName': f"{hmc['VersionInfo']['ServicePackName']}.{hmc['VersionInfo']['Minor']}"
    }
    data.append(row)

df = pd.DataFrame(data)

In [20]:
df

,Name,SerialNumber,Version,ServicePackName
0,dcmipphhmc001.edc.nam.gm.com,215082D,V9R2,953.4
1,dcmipphhmc002.edc.nam.gm.com,215083D,V9R2,953.4
2,dcwipphhmc005.edc.nam.gm.com,215084D,V9R2,953.4
3,dcwipphhmc004.edc.nam.gm.com,215085D,V9R2,953.4
4,dcmirphhmc001.rls.mpg.nam.gm.com,130PLXA,V10R2,1041.2
5,dcmidphhmc002.edc.nam.gm.com,130PPDA,V10R2,1031.1
6,dcwitphhmc001.edc.nam.gm.com,130R9KA,V10R2,1031.1
7,dcwipphhmc006.edc.nam.gm.com,130RGWA,V10R2,1031.1
8,dcmipphhmc003.edc.nam.gm.com,130RH9A,V10R2,1031.1
9,dcmipphhmc006.edc.nam.gm.com,13119VA,V10R2,1031.1
